In [1]:
import pandas as pd
import sqlite3 



df = pd.read_csv("amazon.csv")  
df.columns

Index(['product_id', 'product_name', 'category', 'discounted_price',
       'actual_price', 'discount_percentage', 'rating', 'rating_count',
       'about_product', 'user_id', 'user_name', 'review_id', 'review_title',
       'review_content', 'img_link', 'product_link'],
      dtype='str')

In [2]:
df = pd.read_csv("amazon.csv")
df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]
conn = sqlite3.connect('AmazonDB.db')
df.to_sql('Productos',conn, if_exists='replace', index=False)


1465

In [7]:
consulta = ("SELECT category, AVG(rating), COUNT(*) FROM productos GROUP BY category")
conn = sqlite3.connect('AmazonDB.db')
df = pd.read_sql_query(consulta,conn)
print(df)
conn.close()

                category  AVG(rating)  COUNT(*)
0          Car&Motorbike     3.800000         1
1  Computers&Accessories     4.154967       453
2            Electronics     4.081749       526
3    Health&PersonalCare     4.000000         1
4           Home&Kitchen     4.031696       448
5        HomeImprovement     4.250000         2
6     MusicalInstruments     3.900000         2
7         OfficeProducts     4.309677        31
8             Toys&Games     4.300000         1


In [3]:
conn = sqlite3.connect("AmazonDB.db")
df["category"]= df["category"].str.split('|').str[0]
print(df.columns)
df[["product_id","category"]].to_sql("temp_productos",conn, if_exists="replace", index=False)

Index(['product_id', 'product_name', 'category', 'discounted_price',
       'actual_price', 'discount_percentage', 'rating', 'rating_count',
       'about_product', 'user_id', 'user_name', 'review_id', 'review_title',
       'review_content', 'img_link', 'product_link'],
      dtype='str')


1465

In [4]:
cursor = conn.cursor()
cursor.execute("""
UPDATE Productos  
SET category = (
SELECT temp_productos.category
FROM temp_productos
WHERE temp_productos.product_id = Productos.product_id
)
WHERE EXISTS(
SELECT 1
FROM temp_productos
WHERE temp_productos.product_id = Productos.product_id
)
""")

In [5]:
conn.commit()
cursor.execute("DROP TABLE temp_productos;")
conn.close()

In [6]:
consulta = ("SELECT  category, COUNT(*), AVG(rating) FROM Productos GROUP BY category")
conn = sqlite3.connect("AmazonDB.db")
df = pd.read_sql_query(consulta,conn)
print(df)
conn.close()

                category  COUNT(*)  AVG(rating)
0          Car&Motorbike         1     3.800000
1  Computers&Accessories       453     4.154967
2            Electronics       526     4.081749
3    Health&PersonalCare         1     4.000000
4           Home&Kitchen       448     4.031696
5        HomeImprovement         2     4.250000
6     MusicalInstruments         2     3.900000
7         OfficeProducts        31     4.309677
8             Toys&Games         1     4.300000


In [ ]:

tabla = df.plot(x="category", y="AVG(rating)", kind='bar', figsize=(10,6), ylim=(0,5), title="Promedio de rating por categoria")
tabla.bar_label(tabla.containers[0],padding=3);


    